# Figure 6: Which parts of a human disorder a mouse can reach

Section 5 measured, from mouse connectivity alone, which human cortex has been connectionally
reorganised. If that measurement means anything, it should **predict where a human disorder lies beyond
a mouse model's reach**.

It does, and it does so selectively.

**The claim.** We correlated HOMER **coverage** (log₁₀ mass-normalised mean π mass per region) with
ENIGMA case-control **Cohen's d** across the 30 Desikan–Killiany regions HOMER resolves. A *positive* ρ
means **cortical thinning is more severe where less mouse mass arrives**, so the disorder lives where the
mouse cannot go.

- **Bipolar disorder** ρ = +0.64 (spin p < 0.001)
- **Schizophrenia** ρ = +0.52 (spin p = 0.002)
- **Every one of the other 13 conditions tested: null.**

We then split each disorder by compartment, and this is what makes the result a discovery rather than a
restatement: **within those two disorders the relationship reverses in subcortex.** Their subcortical
signature sits where coverage is *highest*. A mouse cannot reach their cortical signature; it can reach
their subcortical one.

## What is validation and what is discovery

**Validation.** van den Heuvel et al. (*Brain*, 2019) showed that human-specific cortical connectivity
features are implicated in schizophrenia dysconnectivity and **not** in ASD, OCD, MDD, bvFTD or
Alzheimer's. Our cortical selectivity result **replicates that**, reached from *mouse connectivity
alone*, with no human disorder data entering the model. We frame it as a validation. It is not a new
claim.

**Discovery.** The within-disorder cortex/subcortex decomposition is new. It says which *component* of a
disorder a mouse model can address, which is an actionable statement about preclinical study design.

> ### The definitional trap, stated once more
> Coverage must be the **mass-normalised mean**. Summing instead of averaging makes coverage scale with
> the number of parcels a region contains and **abolishes the effect entirely** (ρ = +0.64 → +0.05;
> ED6e). This is not a robustness check we chose to run. It is the difference between the result and a
> null, and an earlier version of this analysis reported the null.

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
LOGS = ROOT / 'outputs' / 'logs'

dd  = json.loads((LOGS / 'section6_double_dissociation.json').read_text())
sel = json.loads((LOGS / 'section6_selectivity_battery.json').read_text())
rob = json.loads((LOGS / 'section6_robustness.json').read_text())

print('atlas   :', dd['_meta']['atlas'])
print('coverage:', dd['_meta']['coverage'])
print('cortex  :', dd['_meta']['cortical_metric'])
print('subcortex:', dd['_meta']['subcortical_metric'])
print('sign    :', dd['_meta']['note'])
print('spins   :', dd['_meta']['n_spin'])

## 1. Selectivity across 15 ENIGMA conditions (Fig. 6b)

This is the panel the section rests on. If coverage predicted thinning in *every* disorder it would be a
generic "association cortex is vulnerable" statement and worth little. It does not.

All 15 conditions, one spin-tested correlation each, FDR-corrected across the battery.

In [ ]:
bat = sel['selectivity_battery']
rows = sorted(bat.items(), key=lambda kv: -kv[1]['spearman'])

print(f"{'condition':<30} {'rho':>7} {'spin p':>8} {'FDR q':>8} {'mean|d|':>8}")
print('-' * 68)
for name, d in rows:
    star = ' *' if d['fdr_q'] < 0.05 else '  '
    print(f"{name:<30} {d['spearman']:>+7.2f} {d['spin_p']:>8.3f} {d['fdr_q']:>8.3f}{star} "
          f"{d['mean_abs_d']:>7.2f}")
print()
sig = [n for n, d in rows if d['fdr_q'] < 0.05]
print(f'survive FDR: {sig}')
print()
print('Only bipolar disorder and schizophrenia. Note the well-powered NULLS. They are what make the')
print('result specific rather than a truism:')
for n in ('22q11 deletion syndrome', 'Normal ageing, 3–29 yr', 'Temporal lobe epilepsy (L)'):
    if n in bat:
        print(f"  {n:<30} rho = {bat[n]['spearman']:+.2f}   mean |d| = {bat[n]['mean_abs_d']:.2f}   "
              f"q = {bat[n]['fdr_q']:.2f}  -> null despite a substantial effect size")
print()
print('22q11 deletion syndrome is the sharpest of these. It is the LARGEST KNOWN GENETIC RISK FACTOR')
print('FOR PSYCHOSIS, and coverage does not flag it. That rules out the lazy reading ("coverage tracks')
print('psychiatric risk"). Coverage indexes ANATOMY rather than diagnostic category. 22q11 does not')
print('share the cortical topography that bipolar disorder and schizophrenia share (ED6c).')

### FDR: which correction?

The q values here come from the **15-condition battery**. An earlier draft quoted q values from a
6-disorder correction. Both disorders survive either way, but the paper's claim rests on the *selectivity
across the full battery*, so the full-battery correction is the one to quote.

In [ ]:
# ---------------- Fig 6b: selectivity ----------------
names = [n for n, _ in rows]
rhos = [d['spearman'] for _, d in rows]
qs = [d['fdr_q'] for _, d in rows]
ds = [d['mean_abs_d'] for _, d in rows]
cols = ['#c1272d' if q < 0.05 else '#c8c8c8' for q in qs]

fig, ax = plt.subplots(figsize=(5.0, 6.0))
y = np.arange(len(names))[::-1]
ax.barh(y, rhos, color=cols, zorder=3, height=0.72)
ax.axvline(0, color='0.3', lw=1)
ax.set_yticks(y)
ax.set_yticklabels([f'{n}   |d| {d_:.2f}' for n, d_ in zip(names, ds)], fontsize=8)
ax.set_xlabel('Spearman ρ (coverage vs cortical-thinning Cohen’s d)')
ax.set_xlim(-0.55, 1.05)
ANNOT_X = 0.70
for yi, (r_, q_) in zip(y, zip(rhos, qs)):
    ax.text(ANNOT_X, yi, f'ρ={r_:+.2f}  q={q_:.3f}', va='center', fontsize=7.2,
            color='#c1272d' if q_ < 0.05 else '0.55')
ax.set_title('Only two disorders lie beyond the mouse’s reach\n'
             f'{len(sig)} of {len(rows)} conditions survive FDR across the battery',
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

## 2. The double dissociation (Fig. 6a, 6c): the discovery

Within bipolar disorder and schizophrenia, split the brain into cortex and subcortex and correlate
coverage with the disorder's effect size **separately in each**.

In [ ]:
print(f"{'':<16} {'cortex':>22}   {'subcortex':>22}   {'interaction':>16}")
print('-' * 84)
for k in ('bipolar', 'schizophrenia'):
    c, s_, i = dd[k]['cortex'], dd[k]['subcortex'], dd[k]['interaction']
    print(f"{k:<16} rho = {c['spearman']:+.2f} (n={c['n']}, p={c['spin_p']:.4f})   "
          f"rho = {s_['spearman']:+.2f} (n={s_['n']})        "
          f"Fisher z p = {i['p']:.3f}")
print()
print('The sign FLIPS. In cortex, the disorder is worse where coverage is LOW (the mouse cannot reach')
print('it). In subcortex, the disorder is worse where coverage is HIGH (the mouse CAN reach it).')
print('The interaction is significant in both disorders.')
print()
print('THE ACTIONABLE STATEMENT: a mouse model cannot address the cortical signature of bipolar')
print('disorder or schizophrenia, but it CAN address their subcortical signature. That is a claim')
print('about which component of a disorder is modellable, rather than about whether the disorder is.')

c = dd['bipolar']['cortex']
print()
print(f"Effect magnitude: {100 * c['burden_low_coverage_tertile']:.0f} % of bipolar disorder's total")
print(f"thinning burden falls in the LEAST-covered third of cortex, against "
      f"{100 * c['burden_high_coverage_tertile']:.0f} % in the best-covered.")

### Three limitations, stated plainly

1. **The subcortical arm is n = 7 regions.** That is a small number, and we say so. The claim rests on
   the interaction test rather than on the subcortical ρ on its own.
2. **The two arms use different metrics.** Cortex is *thickness* Cohen's d; subcortex is *volume*
   Cohen's d. ENIGMA does not publish subcortical thickness. A sign reversal across two different
   measurements is weaker evidence than a sign reversal within one.
3. **No mechanism.** We show a correspondence between connectional reorganisation and disorder anatomy.
   We do not show why. A per-disorder "translatability index" was built and **failed** (Parkinson's scored
   as poorly as schizophrenia, because ENIGMA's subcortical panel has no substantia nigra, so the index
   measures where a disorder is visible to volumetric MRI rather than where it is). It is not reported.

In [ ]:
# ---------------- Fig 6c: the decomposition ----------------
from scipy.stats import rankdata

fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.2))
for ax, k in zip(axes, ('bipolar', 'schizophrenia')):
    sub = dd[k]['subcortex']
    cov_s = np.array(list(sub['coverage'].values()), float)
    d_s = np.array(list(sub['cohens_d'].values()), float)
    # rank axes: raw axes are flattened by outliers, and the reported statistic IS a rank correlation
    ax.scatter(rankdata(cov_s), rankdata(d_s), s=52, color='#1b4f8a', zorder=3,
               edgecolor='white', linewidth=0.6, label=f"subcortex  ρ = {sub['spearman']:+.2f} (n = {sub['n']})")
    b = np.polyfit(rankdata(cov_s), rankdata(d_s), 1)
    xs = np.array([1, len(cov_s)])
    ax.plot(xs, np.polyval(b, xs), color='#1b4f8a', lw=1.6, ls='--')
    cx = dd[k]['cortex']
    ax.set_xlabel('coverage (rank)')
    ax.set_ylabel('disorder effect size (rank)')
    ax.legend(frameon=False, fontsize=8.5, loc='upper right')
    ax.set_title(f"{k}\ncortex ρ = {cx['spearman']:+.2f} (n = {cx['n']}), "
                 f"subcortex ρ = {sub['spearman']:+.2f} (n = {sub['n']}), "
                 f"interaction p = {dd[k]['interaction']['p']:.3f}",
                 fontweight='bold', loc='left', fontsize=9.5)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
fig.subplots_adjust(wspace=0.32, top=0.80)
fig.suptitle('The relationship reverses in subcortex: a mouse can reach one component and not the other',
             fontsize=11.5, fontweight='bold', x=0.02, ha='left', y=1.03)
plt.show()

## 3. Is coverage just the sensorimotor–association axis? (ED6a)

The obvious objection. Published hierarchy maps **do** predict this thinning, which is well established
(Sydnor 2021). The question is whether coverage adds anything.

We are careful here, and we do **not** claim coverage is the better predictor.

In [ ]:
h2h = sel['hierarchy_head_to_head']
for k in ('bipolar', 'schizophrenia'):
    print(f'{k}:')
    for m, d in h2h[k].items():
        star = ' *' if d['spin_p'] < 0.05 else '  '
        print(f"    {m:<22} rho = {d['spearman']:+.2f}   spin p = {d['spin_p']:.3f}{star}")
    print()

ctl = sel['controls']
print('after partialling BOTH hierarchy maps out of coverage:')
for k in ('bipolar', 'schizophrenia'):
    print(f"  {k:<16} rho = {ctl[k]['coverage_partialling_both_hierarchy_maps']:+.2f}")
print()
print('Coverage attains the largest correlation of any predictor tested, is the ONLY map significant in')
print('BOTH disorders, and survives partialling the hierarchy maps out.')
print()
print('WHAT WE DO NOT CLAIM: that coverage BEATS the hierarchy maps. At 30 regions the two are not')
print("statistically distinguishable (Williams' test for dependent correlations, p = 0.33). The panel")
print('establishes that the effect is NOT REDUCIBLE to the sensorimotor–association axis. It does not')
print('establish that coverage outperforms it, and we do not say that it does.')

## 4. The controls that could have killed it (ED6b, d, e)

### The anchor-distance confound
The coupling is anchored on 42 curated homologous regions. Low coverage might merely index *distance
from the anchor set*, an artefact of where we happened to put supervision. Anchor distance **does**
correlate with coverage, and it **does** predict bipolar thinning on its own. The association survives
adjustment anyway.

### The parcel-count control
Summed rather than mass-normalised coverage. This is the one that matters most.

In [ ]:
ad = rob['anchor_distance']
print(f"coverage vs anchor distance: rho = {ad['coverage_vs_anchor_distance']:+.2f}")
for k in ('bipolar', 'schizophrenia'):
    print(f'  {k}: {json.dumps(ad[k])}')
print()

print('robustness across every analysis choice (bipolar):')
for k, v in rob['robustness']['bipolar'].items():
    print(f'  {k:<28} rho = {v:+.2f}')
print()
print('No analysis choice changes the conclusion.')
print()

sc = rob['summed_coverage_control']
print('THE PARCEL-COUNT CONTROL: mass-normalised MEAN vs SUM:')
for k in ('bipolar', 'schizophrenia'):
    print(f"  {k:<16} mass-normalised rho = {sc[k]['mass_normalised']:+.2f}   "
          f"summed rho = {sc[k]['summed']:+.2f}")
print()
print('Summing abolishes the effect entirely. The mass-normalisation is NOT a free parameter and NOT a')
print('post-hoc choice: summing introduces a parcel-count confound, which is a parcellation artefact.')
print('This is the bug an earlier version of the analysis had, and it reported a null.')

In [ ]:
# ---------------- ED6e: the parcel-count control ----------------
fig, ax = plt.subplots(figsize=(5.2, 3.8))
x = np.arange(2)
w = 0.36
mn = [sc['bipolar']['mass_normalised'], sc['schizophrenia']['mass_normalised']]
sm = [sc['bipolar']['summed'], sc['schizophrenia']['summed']]
ax.bar(x - w / 2, mn, w, color='#c1272d', label='mass-normalised mean (correct)', zorder=3)
ax.bar(x + w / 2, sm, w, color='#c8c8c8', label='summed (parcel-count confounded)', zorder=3)
for xi, (a_, b_) in enumerate(zip(mn, sm)):
    ax.text(xi - w / 2, a_ + 0.02, f'{a_:+.2f}', ha='center', fontsize=9)
    ax.text(xi + w / 2, b_ + 0.02, f'{b_:+.2f}', ha='center', fontsize=9)
ax.axhline(0, color='0.3', lw=1)
ax.set_xticks(x)
ax.set_xticklabels(['bipolar disorder', 'schizophrenia'])
ax.set_ylabel('Spearman ρ (coverage vs Cohen’s d)')
ax.set_ylim(-0.05, 0.78)
ax.legend(frameon=False, fontsize=8.5)
ax.set_title('The mass-normalisation is not a free parameter\n'
             'summing instead of averaging abolishes the relationship entirely',
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

## 5. Summary

| finding | value |
|---|---|
| bipolar disorder, cortex | ρ = +0.64, spin p < 0.001, FDR q = 0.004 |
| schizophrenia, cortex | ρ = +0.52, spin p = 0.002, FDR q = 0.004 |
| the other 13 conditions | all null |
| 22q11 deletion (well-powered null) | ρ ≈ −0.1, \|d\| = 0.39 |
| bipolar disorder, subcortex | ρ = −0.68 (n = 7) |
| schizophrenia, subcortex | ρ = −0.79 (n = 7) |
| cortex × subcortex interaction | Fisher z p = 0.003 / 0.002 |
| summed-coverage control | ρ = +0.05 / +0.02 (**effect gone**) |

**Validation:** the cortical selectivity replicates van den Heuvel et al. (2019), who found that
human-specific cortical organisation is implicated in schizophrenia and not in most other conditions.
We reach it here from mouse connectivity alone.

**Discovery:** the cortex/subcortex reversal says *which component* of a disorder a mouse model can
address. For bipolar disorder and schizophrenia, the answer is: not the cortical signature, but yes the
subcortical one.

### Panels not produced here

Fig. 6a (the cortical scatter on DK regions) is built by `fig6/make_fig6.py`; ED6b/d/e by
`fig_6_ED/make_ed6_controls.py`.